In [ ]:
import numpy as np
import scipy as sp
import scipy.sparse as sparse
import matplotlib.pyplot as plt

### Acknowledgement

I would like to thank Scott MAcLachlan from Memorial University for inspiration: this tutorial is inspired by a multigrid tutorial he gave (online) at a summer school during covid at the Laval University.

### 1D Poisson ($u_{xx} = 0 \; \& \; u(0)=u(1)=0$)

Let's set up a matrix
$$
L = 
\frac{1}{h^2} \left[
\begin{array}{r r r r r}
2 & -1 &  & & \\
-1 & 2 & -1 & & \\
   & -1 & 2 & \ddots \\
& & \ddots & & -1 \\
& & & -1 & 2
\end{array}
\right],

\qquad h = 1/(n+1).
$$


The eigenpairs of $L$ are known explicitly - the discrete Fourier modes/frequencies:


$$
L \mathbf{w}_k = \lambda_k \mathbf{w}_k 
\qquad \mathrm{with} \qquad 
(\mathbf{w}_k)_j = \sin\left(\frac{j \cdot k\pi}{n+1}\right)
\quad \mathrm{and} \quad
\lambda_k = \frac{4}{h^2} \sin^{2} \left(\frac{k \pi}{2(n+1)}\right)
$$

In [ ]:
m = 5
n = 2**m-1; h = 1/(n+1)
L = 1/h**2 * ( np.diag(-1*np.ones(n-1),-1) + np.diag(2*np.ones(n)) + np.diag(-1*np.ones(n-1),1) )

# eigpairs
evals, evecs = np.linalg.eigh(L)

# sort from smallest to largest
inds = np.argsort(evals)
evals_L = evals[inds]
evecs_L = evecs[:,inds]

plt.figure(figsize=(12,6))
k_totry = [0,1,12]
x = np.linspace(0,1,n+2)[1:-1]
for ind_k in range(0,len(k_totry)):
    k = k_totry[ind_k]
    line, = plt.plot(x,evecs_L[:,k], lw=2)
    plt.xlabel('x', fontsize=10);
    c = plt.getp(line,'color')
    plt.text(x[k], evecs_L[k,k], f'{evals_L[k]:4.2}', color=c, bbox=dict(facecolor='w'), fontsize=13)
plt.title(r'$\mathrm{eigenvectors} \; L$',fontsize=13); plt.show()

### Jacobi stationary method

The Jacobi stationary method gives the iteration
$$
\mathbf{u}_{k} = \mathbf{u}_{k-1} + D^{-1}(\mathbf{b} - L\mathbf{u}_{k-1})
$$
and has the iteration matrix 
$$ E = I - D^{-1}L  = I - \frac{1}{2} 
\left[
\begin{array}{r r r r r}
2 & -1 &  & & \\
-1 & 2 & -1 & & \\
   & -1 & 2 & -1 & \\
& & \ddots & & \\
& & & -1 & 2
\end{array}
\right],
$$  

Hence, the eigenvectors of $L$ and $E$ coincide. The eigenvalues of $E$ become

$$
E \mathbf{w}_k = \left( 1 - 2 \sin^{2} \left(\frac{k \pi}{2(n+1)}\right) \right) \mathbf{w}_k .
$$

In [ ]:
E = np.eye(n)  -  h**2/2 * L

# eigpairs
evals, evecs = np.linalg.eig(E)

# sort from smallest to largest
inds = np.argsort(evals)
evals_E = evals[inds]
evecs_E = evecs[:,inds]


plt.figure(figsize=(12,6))
k_totry = [-1,-2,-12]
x = np.linspace(0,1,n+2)[1:-1]
for ind_k in range(0,len(k_totry)):
    k = k_totry[ind_k]
    line, = plt.plot(x,evecs_E[:,k], lw=2)
    plt.xlabel('x', fontsize=10);
    c = plt.getp(line,'color')
    plt.text(x[k], evecs_E[k,k], f'{evals_E[k]:4.2}', color=c, bbox=dict(facecolor='w'), fontsize=13)
plt.title(r'$\mathrm{eigenvectors} \; E$',fontsize=13); plt.show()

In [ ]:
plt.figure(figsize=(14,7))
plt.subplot(1,2,1)
plt.plot(evals_L, 0*evals_L, 'o', c='tab:red', ms=10, markeredgecolor='k', label=r'$\lambda_L$')
plt.yticks([])
plt.xlabel(r'$\mathrm{eigenvalues} \; L$',fontsize=13); plt.legend(fontsize=14); 

plt.subplot(1,2,2)
plt.plot(1-h**2*0.5*evals_L, 0*evals_L, 'o', c='tab:green', ms=17, markeredgecolor='k', alpha=.4, label=r'$1-\frac{h^2}{2}\cdot\lambda_L$')
plt.plot(evals_E, 0*evals_E, 'o', c='tab:blue', ms=9, markeredgecolor='k', label=r'$\lambda_E$')

plt.yticks([])
plt.xlabel(r'$\mathrm{eigenvalues} \; E$',fontsize=13)

plt.legend(fontsize=14); plt.show()

In [ ]:
k_totry = [2,10,20,31]; nu = 100
u1 = np.sin(np.arange(1,n+1) * k_totry[0] * np.pi / (n+1)); enorm1 = []
u2 = np.sin(np.arange(1,n+1) * k_totry[1] * np.pi / (n+1)); enorm2 = []
u3 = np.sin(np.arange(1,n+1) * k_totry[2] * np.pi / (n+1)); enorm3 = []
u4 = np.sin(np.arange(1,n+1) * k_totry[3] * np.pi / (n+1)); enorm4 = []

u = u1
for i in range(nu):
    u = E @ u
    enorm1.append(np.linalg.norm(u))

u = u2
for i in range(nu):
    u = E @ u
    enorm2.append(np.linalg.norm(u))

u = u3
for i in range(nu):
    u = E @ u
    enorm3.append(np.linalg.norm(u))

u = u4
for i in range(nu):
    u = E @ u
    enorm4.append(np.linalg.norm(u))

plt.figure(figsize=(10,6))
for rnorm, label in zip([enorm1, enorm2, enorm3, enorm4],
                        ['k={}'.format(k_totry[0]), 'k={}'.format(k_totry[1]), 'k={}'.format(k_totry[2]), 'k={}'.format(k_totry[3])]):
    rnorm = np.array(rnorm) / rnorm[0]
    plt.semilogy(rnorm, lw=3, label=label)
plt.xlabel('iterations', fontsize=13)
plt.title('error norms', fontsize=13)
plt.legend(fontsize=14); plt.show()

### Damped/Relaxed Jacobi stationary method

The Jacobi stationary method has the iteration matrix 
$$ E_{\alpha} = I - \alpha D^{-1}L  = I - \frac{\alpha}{2} 
\left[
\begin{array}{r r r r r}
2 & -1 &  & & \\
-1 & 2 & -1 & & \\
   & -1 & 2 & -1 & \\
& & \ddots & & \\
& & & -1 & 2
\end{array}
\right],
$$  

Hence, the eigenvectors of $L$ and $E_{\alpha}$ coincide. The eigenvalues of $E_{\alpha}$ become

$$
E_{\alpha} \mathbf{w}_k = \left( 1 - 2 \alpha \sin^{2} \left(\frac{k \pi}{2(n+1)}\right) \right) \mathbf{w}_k .
$$

In [ ]:
alpha = 2/3
E_alpha = np.eye(n)  -  alpha * h**2/2 * L

# eigpairs
evals, evecs = np.linalg.eig(E_alpha)

# sort from smallest to largest
inds = np.argsort(evals)
evals_Ealpha = evals[inds]


plt.figure(figsize=(22,8))
plt.subplot(1,3,1)
plt.plot(evals_L, 0*evals_L, 'o', ms=9, c='tab:red', markeredgecolor='k', label=r'$\lambda_L$')
plt.yticks([])
plt.xlabel(r'$\mathrm{eigenvalues} \; L$',fontsize=13); plt.legend(fontsize=14); 

plt.subplot(1,3,2)
plt.plot(1-h**2*0.5*evals_L, 0*evals_L, 'o', c='tab:green', ms=17, markeredgecolor='k', alpha=.4, label=r'$1-\frac{h^2}{2}\cdot\lambda_L$')
plt.plot(evals_E, 0*evals_E, 'o', c='tab:blue', markeredgecolor='k', ms=9, label=r'$\lambda_E$')
plt.yticks([])
plt.xlabel(r'$\mathrm{eigenvalues} \; E$',fontsize=13); plt.legend(fontsize=14); 

plt.subplot(1,3,3)
plt.plot(1-alpha*h**2*0.5*evals_L, 0*evals_L, 'o', c='white', ms=17, alpha=.4, label=r'$1-\alpha \frac{h^2}{2}\cdot\lambda_L$')
plt.plot(1-alpha*h**2*0.5*evals_L, 0*evals_L, 'o', c='tab:green', ms=17, markeredgecolor='k', alpha=.4, label=r'$1-\alpha \frac{h^2}{2}\cdot\lambda_L$')
plt.plot(evals_Ealpha, 0*evals_Ealpha, 'o', c='tab:blue', ms=9, markeredgecolor='k', label=r'$\lambda_{\alpha}$')
plt.yticks([]); plt.xlim([-1,1.1])
plt.xlabel(r'$\mathrm{eigenvalues} \; E_{\alpha}, \; \alpha = 2/3$',fontsize=13); plt.legend(fontsize=14); 

plt.show()

In [ ]:
k_totry = [1,10,20,31]; nu = 100
u1 = np.sin(np.arange(1,n+1) * k_totry[0] * np.pi / (n+1)); enorm1 = []
u2 = np.sin(np.arange(1,n+1) * k_totry[1] * np.pi / (n+1)); enorm2 = []
u3 = np.sin(np.arange(1,n+1) * k_totry[2] * np.pi / (n+1)); enorm3 = []
u4 = np.sin(np.arange(1,n+1) * k_totry[3] * np.pi / (n+1)); enorm4 = []

alpha = 2/3
E_alpha = np.eye(n)  -  alpha * h**2/2 * L

u = u1
for i in range(nu):
    u = E_alpha @ u
    enorm1.append(np.linalg.norm(u))

u = u2
for i in range(nu):
    u = E_alpha @ u
    enorm2.append(np.linalg.norm(u))

u = u3
for i in range(nu):
    u = E_alpha @ u
    enorm3.append(np.linalg.norm(u))

u = u4
for i in range(nu):
    u = E_alpha @ u
    enorm4.append(np.linalg.norm(u))

plt.figure(figsize=(10,6))
for rnorm, label in zip([enorm1, enorm2, enorm3, enorm4],
                        ['k={}'.format(k_totry[0]), 'k={}'.format(k_totry[1]), 'k={}'.format(k_totry[2]), 'k={}'.format(k_totry[3])]):
    rnorm = np.array(rnorm) / rnorm[0]
    plt.semilogy(rnorm, lw=3, label=label)
plt.xlabel('iterations', fontsize=13)
plt.title('error norms', fontsize=13)
plt.legend(fontsize=14); plt.show()

In [ ]:
plt.figure(figsize=(10,6))
K = np.arange(1,n)

alpha = 1.0 / 3.0
lmbda = 1 - (alpha / 2) * 4.0 * np.sin(np.pi * K / (2*(n+1)))**2
plt.plot(lmbda,'-',c='tab:blue',label='residual',linewidth=3, clip_on=False)
plt.text(n+2, lmbda[-1], r'$\alpha=1/3$',fontsize=13)

alpha = 1.0 /2
lmbda = 1 - (alpha / 2) * 4.0 * np.sin(np.pi * K / (2*(n+1)))**2
plt.plot(lmbda,'-',c='tab:blue',label='residual',linewidth=3, clip_on=False)
plt.text(n+2, lmbda[-1], r'$\alpha=1/2$',fontsize=13)

alpha = 2.0 / 3.0
lmbda = 1 - (alpha / 2) * 4.0 * np.sin(np.pi * K / (2*(n+1)))**2
plt.plot(lmbda,'-',c='tab:blue',label='residual',linewidth=3, clip_on=False)
plt.text(n+2, lmbda[-1], r'$\alpha=2/3$',fontsize=13)

alpha = 1.
lmbda = 1 - (alpha / 2) * 4.0 * np.sin(np.pi * K/ (2*(n+1)))**2
plt.plot(lmbda,'-',c='tab:blue',label='residual',linewidth=3, clip_on=False)
plt.text(n+2, lmbda[-1], r'$\alpha=1$',fontsize=13)

plt.plot([0, n], [0, 0], '--', c='tab:gray')
plt.axis([0,n,-1,1])
plt.ylabel(r'$\lambda^{\alpha}_k$',fontsize=13)
plt.xlabel(r'eigenvalue index, $k$',fontsize=13)
plt.title(r'Eigenvalue distribution for various $\alpha$',fontsize=13); plt.show()

### Damped Jacobi as a smoother

In [ ]:
k_totry = [1,10,20,31]
v1 = np.sin(np.arange(1,n+1) * k_totry[0] * np.pi / (n+1))
v2 = np.sin(np.arange(1,n+1) * k_totry[1] * np.pi / (n+1))
v3 = np.sin(np.arange(1,n+1) * k_totry[2] * np.pi / (n+1))
v4 = np.sin(np.arange(1,n+1) * k_totry[3] * np.pi / (n+1))

alpha_1 = 1.
alpha_2 = 2.0 / 3.0
nu = 3

e_init = (v1 + v2 + v3 + v4)/4

e_alpha1 = e_init
E_alpha1 = np.eye(n)  -  alpha_1 * h**2/2 * L
for i in range(50):
    e_alpha1 = E_alpha1 @ e_alpha1
    if i == nu-1:
        e_alpha1_aftrnu = e_alpha1

e_alpha2 = e_init
E_alpha2 = np.eye(n)  -  alpha_2 * h**2/2 * L
for i in range(50):
    e_alpha2 = E_alpha2 @ e_alpha2
    if i == nu-1:
        e_alpha2_aftrnu = e_alpha2


plt.figure(figsize=(20,7))
x = np.linspace(0,1,n+2)[1:-1]; K = np.arange(1,n)
plt.subplot(1,3,1)
plt.plot(x, e_init, '--', c='tab:gray', markeredgecolor='k', label=r'initial error')
plt.plot(x, e_alpha1_aftrnu, '-o', c='orange', markeredgecolor='k', label=r'error after 3 iterations')
plt.plot(x, e_alpha1, '-o', c='tab:red', markeredgecolor='k', label=r'error after 100 iterations')
plt.xlabel(r'$x$',fontsize=13); plt.legend(fontsize=14); plt.title(r'$\alpha = 1$') 

plt.subplot(1,3,2)
plt.plot(x, e_init, '--', c='tab:gray', markeredgecolor='k', label=r'initial error')
plt.plot(x, e_alpha2_aftrnu, '-o', c='tab:green', markeredgecolor='k', label=r'error after 3 iterations')
plt.plot(x, e_alpha2, '-o', c='tab:blue', markeredgecolor='k', label=r'error after 100 iterations')
plt.xlabel(r'$x$',fontsize=13); plt.legend(fontsize=14); plt.title(r'$\alpha = 2/3$')

plt.subplot(1,3,3)
lmbda = 1 - (alpha_1 / 2) * 4.0 * np.sin(np.pi * K / (2*(n+1)))**2
plt.plot(lmbda,'-',c='tab:red',label='residual',linewidth=3, clip_on=False)
plt.text(n+2, lmbda[-1], r'$\alpha=1$',fontsize=13)
lmbda = 1 - (alpha_2 / 2) * 4.0 * np.sin(np.pi * K/ (2*(n+1)))**2
plt.plot(lmbda,'-',c='tab:blue',label='residual',linewidth=3, clip_on=False)
plt.text(n+2, lmbda[-1], r'$\alpha=2/3$',fontsize=13)

plt.plot([0, n], [0, 0], '--', c='tab:gray'); plt.axis([0,n,-1,1])
plt.ylabel(r'$\lambda^{\alpha}_k$',fontsize=13)
plt.xlabel(r'eigenvalue index, $k$',fontsize=13)
plt.title(r'Eigenvalue distribution for various $\alpha$',fontsize=13)
plt.show()

### Eigenvector aliasing

We can easily verify that

$$
(\mathbf{w}_k^{(n)})_{2j} = \sin\left(\frac{2j \cdot k\pi}{n+1}\right) = \sin\left(\frac{j \cdot k\pi}{(n+1)/2}\right) = (\mathbf{w}_k^{(n/2)})_{j}
\qquad \mathrm{for} \; k < n/2
$$

This formula shows the following: the *smooth* modes ($k$ small compared to $n$) are *preserved* and become *less smooth* ($k$ becomes bigger compared to $n$) if we replace $n$ with $n/2$.

As a result, the *rather smooth* error components in $\mathbf{e}^{(n)}$ become *rather oscilatory* if we move to $\mathbf{e}^{(n/2)}$ and thus could be efficiently treated by Jacobi there (or solved for directly). 

In [ ]:
nf = 2**m - 1
xfull_f = np.linspace(0,1,nf+2); x_f = xfull_f[1:-1]
nc = 2**(m-1) - 1
xfull_c = np.linspace(0,1,nc+2); x_c = xfull_c[1:-1]

plt.figure(figsize=(15,5))
plt.plot(xfull_c, 0*xfull_c, 's', color='tab:orange', markeredgecolor='k', ms=15, label='coarse')
plt.plot(xfull_f, 0*xfull_f, 'o', color='tab:blue', markeredgecolor='k', label='fine')
plt.xlabel(r'$x$',fontsize=13); plt.legend(fontsize=14); plt.title(r'Coarsening by factor 2'); plt.show()

In [ ]:
k = 8
yf = np.sin(k * np.pi * xfull_f)
yc = np.sin(k * np.pi * xfull_c)

plt.figure(figsize=(15,5))
plt.plot(xfull_c, 0*xfull_c, 's', color='tab:gray', markeredgecolor='k', alpha=.7, ms=13, label='coarse')
plt.plot(xfull_f, 0*xfull_f, 'o', color='tab:gray', markeredgecolor='k', alpha=.7, label='fine')
plt.plot(xfull_c, yc, '-s', color='tab:orange', markeredgecolor='k', lw=4, alpha=.8, clip_on=False, label='coarse')
plt.plot(xfull_f, yf, '-o', color='tab:blue', markeredgecolor='k', lw=2, clip_on=False, label='fine')
plt.xlabel(r'$x$',fontsize=13); plt.legend(fontsize=14); plt.title(r'effect of coarsening on a smooth mode'); plt.show()

### Error equation

We are in the situation of having an approximation $\mathbf{u}^{(n)}$ of the true solution $\mathbf{u}$. To improve, we would like to solve the correction equation for the error:

$$
L \mathbf{e}^{(n)} = \mathbf{r}^{(n)}
\qquad \mathrm{where} \quad \mathbf{r}^{(n)} := \mathbf{f} - L \mathbf{u}^{(n)},
$$

and then obtain our solution as $\mathbf{u} = \mathbf{u}^{(n)} + \mathbf{e}^{(n)}$. 

Moreover, we know that the error $\mathbf{e}^{(n)}$ is dominated by the so-called "*smooth*" (also called "*low-frequency*" or "*non-scilatory*") components, corresponding to $k$ small.

### Coarsening

By the above, we deduce that the following makes sense:

- *Restriction*: $ \mathbf{r}^{(n/2)} := R_{n}^{n/2} \mathbf{r}^{(n)}; $

- the error on the coarse grid, $\mathbf{e}^{(n/2)}$, then satisfies $ L^{(n/2)} \mathbf{e}^{(n/2)} = \mathbf{r}^{(n/2)} $
    - we can (aproximately) solve this (with Jacobi) to obtain (approximate) $\mathbf{e}^{(n/2)}$

- *Prolongation* $ \mathbf{e}^{(n)} \approx \tilde{\mathbf{e}}^{(n)} := P_{n/2}^{n} \mathbf{e}^{(n/2)} $;

Hence, $\mathbf{u}^{(n)} + \tilde{\mathbf{e}}^{(n)}$ is expected to be a better approximation than $\tilde{\mathbf{u}}^{(n)}$.



### Two-grid method

In [ ]:
def Jacobi(L, u, f, nu):
    n = L.shape[0]
    Dinv = 1.0 / (2.0 * (n+1)**2)
    omega = 2.0 / 3.0
    for steps in range(nu):
        u += omega * Dinv * (f - L @ u)

def interpolate(uc):
    """interpolate u of size 2**(k-1)-1 to 2**(k)-1"""
    nc = len(uc)
    nf = 2**(int(np.log2(nc+1))+1)-1
    uf = np.zeros((nf,))
    I = np.arange(1,nf,2)
    uf[I-1] = 0.5 * uc
    uf[I] = uc
    uf[I+1] += 0.5 * uc
    return uf

def restrict(uf):
    """restrict u of size 2**(k)-1 to 2**(k-1)-1"""
    nf = len(uf)
    nc = 2**(int(np.log2(nf+1))-1)-1
    uc = np.zeros((nc,))
    I = np.arange(1,nf,2)
    uc = 0.25 * uf[I-1] + 0.5 * uf[I] + 0.25 * uf[I+1]
    return uc

def Poisson1D(n):
    """define the 1D Poisson stencil"""
    L = 1/(1/(n+1))**2 * sparse.diags([-1, 2, -1], [-1, 0, 1], shape=(n,n), format='csr')
    return L

def hnorm(r):
    """define ||r||_h = h ||r||_2"""
    n = len(r)
    h = 1.0 / (n+1)
    hrnorm = h * np.linalg.norm(r)
    return hrnorm

In [ ]:
L = Poisson1D(n)
u_init = np.random.rand(n); u = np.copy(u_init)
u_star = np.zeros(n)
f = L @ u_star
nu = 3

print('Initial error ~ ',hnorm(u_star-u_init))

Jacobi(L, u, f, nu); u_AftrPreSmooth = np.copy(u)

r_coarse = restrict(f - L @ u)
L_coarse = Poisson1D(len(r_coarse))
e_coarse = sparse.linalg.spsolve(L_coarse, r_coarse)
ef = interpolate(e_coarse)
u = u + ef; u_AftrCoarsCrrct = np.copy(u)

Jacobi(L, u, f, nu); u_AftrPostSmooth = np.copy(u)




plt.figure(figsize=(22,12))
xfull_f = np.linspace(0,1,nf+2); x_f = xfull_f[1:-1]
nc = 2**(m-1) - 1
xfull_c = np.linspace(0,1,nc+2); x_c = xfull_c[1:-1]
plt.subplot(2,3,1)
plt.plot(x_f, u_star - u_init, '--', c='tab:gray', ms=9, markeredgecolor='k', label=r'initial error')
plt.plot(x_f, u_star - u_AftrPreSmooth, '-o', c='orange', ms=9, markeredgecolor='k', label=r'error after pre-smooth')
plt.xlabel(r'$x$',fontsize=13); plt.legend(fontsize=14); plt.title(r'error ~ {}'.format(hnorm(u_star - u_AftrPreSmooth)), fontsize=14) 

plt.subplot(2,3,2)
plt.plot(x_f, u_star - u_AftrPreSmooth, '--', c='tab:gray', ms=9, markeredgecolor='k', label=r'initial error')
plt.plot(x_f, ef, '-o', c='tab:red', markeredgecolor='k', ms=9, label=r'update obtained in coarse correction')
plt.plot(x_f, u_star - u_AftrCoarsCrrct, '-o', c='orange', ms=9, markeredgecolor='k', label=r'error after coarse correction')
plt.xlabel(r'$x$',fontsize=13); plt.legend(fontsize=14); plt.title(r'error ~ {}'.format(hnorm(u_star - u_AftrCoarsCrrct)), fontsize=14) 

plt.subplot(2,3,3)
plt.plot(x_f, u_star - u_AftrCoarsCrrct, '-o', c='tab:gray', ms=9, markeredgecolor='k', label=r'error after coarse correction')
plt.plot(x_f, u_star - u_AftrPostSmooth, '-o', c='orange', ms=9, markeredgecolor='k', label=r'error after post-smooth')
plt.xlabel(r'$x$',fontsize=13); plt.legend(fontsize=14); plt.title(r'error ~ {}'.format(hnorm(u_star-u_AftrPostSmooth)), fontsize=14)

plt.show()